# Warm start on the IDAES CSTR

Under closed-loop control the next problem is the last one moved one
step, so the last solution moved one step is nearly its answer.
`drto.warm_start_dynamic` shifts it. This notebook runs one loop
iteration on the pattern a loop actually uses: one persistent model,
built and scaled once, then solved, shifted, and solved again, with
the solver's warm-start options at the call site. The shift carries
values only; the solver rebuilds its own multipliers from a good
starting point in its first iteration, faster than any multipliers we
could hand it. The solver is ipopt, which reads the scaling suffix and
the options through its standard interface.

## One model, built and scaled once

Declarations, the setpoint, the terminal segment, the cold start, and
the assembly, then `drto.scale` writing the factors once from the
flowsheet's stated magnitudes, energy in joules near `1e7` and duties
in watts near `1e6`. The whole loop keeps this one model, and every
solve receives the factors under `nlp_scaling_method=user-scaling`.

In [ ]:
import contextlib, io, time

import pyomo.environ as pyo

import drto
from models.idaes_cstr import DC_START, F_IN, VOLUME, build

m = build()

ss = pyo.TransformationFactory("drto.steady_state_simulation").create_using(
    m, controls={m.fs.cstr.control_volume.heat.name: 0.0,
                 m.fs.cstr.inlet.flow_vol.name: F_IN})
drto.initialize_steady_state(ss)
drto.scaled_solve(ss)
cvs = ss.fs.cstr.control_volume
for j, ssp in (("NaOH", m.ss_naoh), ("EthylAcetate", m.ss_ea),
               ("SodiumAcetate", m.ss_sa), ("Ethanol", m.ss_etoh)):
    ssp.set_value(pyo.value(cvs.material_holdup["Liq", j]))
for j, sgn in (("NaOH", 1), ("EthylAcetate", 1),
               ("SodiumAcetate", -1), ("Ethanol", -1)):
    m.mat0[j] = pyo.value(cvs.material_holdup["Liq", j]) + sgn * DC_START * VOLUME
for k in m.eng_ss:
    m.eng_ss[k] = pyo.value(cvs.energy_holdup[k])

pyo.TransformationFactory("drto.infinite_horizon").apply_to(m)
drto.cold_start_dynamic(m, profile="exponential", time_constant=3.0)
pyo.TransformationFactory("drto.dynamic_optimization").apply_to(m)

drto.scale(m, source={"J": 1e7, "W": 1e6})
res = pyo.SolverFactory("ipopt").solve(
    m, options={"nlp_scaling_method": "user-scaling"}, tee=True)
print(res.solver.termination_condition)

## One step later: shift everything

The loop implements the first move and the state advances one sample;
the model's own solution at t = h stands in for the measurement, read
directly, since the model never leaves its own units. The shift moves
every variable one sampling time forward.

In [ ]:
cv = m.fs.cstr.control_volume
h = 1.0
for j in ("NaOH", "EthylAcetate", "SodiumAcetate", "Ethanol"):
    m.mat0[j] = pyo.value(cv.material_holdup[h, "Liq", j])
m.eng0["Liq"] = pyo.value(cv.energy_holdup[h, "Liq"])

print(drto.warm_start_dynamic(m))

## The second solve, warm

The same model again, the solver told to trust the start: the shifted
values as the initial point, the barrier already small, the bound
pushes tiny so the active set stays put. These are solve-call options,
not part of the shift.

In [ ]:
res = pyo.SolverFactory("ipopt").solve(m, options={
    "nlp_scaling_method": "user-scaling",
    "warm_start_init_point": "yes",
    "mu_init": 1e-6,
    "warm_start_bound_push": 1e-9,
    "warm_start_mult_bound_push": 1e-9,
}, tee=True)
print(res.solver.termination_condition)

The cold solve above took seventeen iterations; the
warm-started one lands in single digits: the shifted solution is nearly
the answer, and the solve rebuilds its multipliers from it and stops.
That is warm starting a receding horizon, whole.